# Clase 4: clasificación lineal y regresión logística

Visualizaciones interactivas para acompañar las notas de la clase 4. Las figuras siguen el orden y los colores de las notas (clase 0: círculos azules, clase 1: triángulos rojos) y usan los mismos datos: se importan desde `notes/scripts/04-clasificacion-figuras.py`, el script que genera las figuras del PDF. La única excepción es el ejemplo de datos circulares de la última sección, que no está en las notas.

En las figuras con fondo de color, el color indica la probabilidad $h_\theta(\mathbf{x})$ que el modelo le asigna a la clase 1: rojo cerca de 1, azul cerca de 0 y blanco cerca de 0.5.

Para usarlo en clase basta con ejecutar todas las celdas al inicio y luego mover los sliders.

In [1]:
%matplotlib inline
import importlib.util
from functools import lru_cache
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import interact, widgets
from sklearn.linear_model import LogisticRegression

# Datos y funciones de las notas (los mismos que generan las figuras del PDF)
ruta = Path("../notes/scripts/04-clasificacion-figuras.py")
spec = importlib.util.spec_from_file_location("fig04", ruta)
fig04 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(fig04)

C0, C1, VERDE, MORADO, GRIS = "#1f77b4", "#d62728", "tab:green", "tab:purple", "0.55"
sigmoide = fig04.sigmoide

plt.rcParams.update({
    "figure.figsize": (11, 4.6),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 12,
})


def puntos(ax, X, y, tam=110):
    """Clase 0: círculos azules. Clase 1: triángulos rojos. Grandes, para el proyector."""
    ax.scatter(X[y == 0, 0], X[y == 0, 1], s=tam, marker="o", color=C0, ec="white", lw=1.2, zorder=3, label="$y = 0$")
    ax.scatter(X[y == 1, 0], X[y == 1, 1], s=tam * 1.15, marker="^", color=C1, ec="white", lw=1.2, zorder=3, label="$y = 1$")


def fondo(ax, prob, xlim, ylim, n=250):
    """Colorea el plano según prob(X) = h(x) y dibuja la frontera h = 0.5."""
    g1, g2 = np.meshgrid(np.linspace(*xlim, n), np.linspace(*ylim, n))
    P = prob(np.column_stack([g1.ravel(), g2.ravel()])).reshape(g1.shape)
    ax.imshow(P, extent=(*xlim, *ylim), origin="lower", cmap="RdBu_r", vmin=0, vmax=1, alpha=0.35, aspect="auto")
    ax.contour(g1, g2, P, levels=[0.5], colors="black", linewidths=2)
    ax.set(xlim=xlim, ylim=ylim)


def recta(ax, theta, c=0.0, xlim=(0, 5), **estilo):
    """Dibuja la recta θ_0 + θ_1 x_1 + θ_2 x_2 = c."""
    x1 = np.array(xlim)
    ax.plot(x1, (c - theta[0] - theta[1] * x1) / theta[2], **estilo)

## 1. ¿Por qué no usar regresión lineal? (Figura 1)

Los tumores de las notas: $x$ es el tamaño y $y = 1$ indica que es maligno. La recta verde es el ajuste de mínimos cuadrados, y la curva morada es una regresión logística ajustada a los mismos datos. Las líneas verticales marcan dónde cada modelo cruza 0.5, es decir, su umbral de clasificación.

Qué observar al agregar tumores grandes y alejarlos:

- La recta se inclina para acercarse a los tumores grandes, su umbral se mueve a la derecha y empieza a clasificar mal tumores malignos.
- La logística casi no se mueve: los tumores grandes ya tienen probabilidad cercana a 1, su aporte a $\ell(\theta)$ es casi cero y no tiran de la frontera.

Estos datos son separables, así que la logística se ajusta con un poco de regularización ($\lambda = 0.1$); sin ella, los parámetros crecerían sin límite (sección 4.3).

In [2]:
x_base, y_base = fig04.x_a, fig04.y_a
SEPARACION_LEJOS = 2.5  # distancia entre los tumores grandes


def datos_tumores(n_lejos, posicion):
    x_lejos = posicion + SEPARACION_LEJOS * np.arange(n_lejos)
    return np.concatenate([x_base, x_lejos]), np.concatenate([y_base, np.ones(n_lejos)])


def ver_tumores(n_lejos=0, posicion=24.0):
    x, y = datos_tumores(n_lejos, posicion)
    t0, t1 = fig04.recta_mc(x, y)
    umbral_mc = (0.5 - t0) / t1
    log = LogisticRegression(C=10).fit(x[:, None], y)  # C = 1/λ
    umbral_log = -log.intercept_[0] / log.coef_[0, 0]
    errores_mc = int(np.sum((t0 + t1 * x >= 0.5) != (y == 1)))
    errores_log = int(np.sum(log.predict(x[:, None]) != y))

    x_max = max(10, x.max() + 2)
    xs = np.linspace(0, x_max, 500)
    fig, ax = plt.subplots(figsize=(12, 4.8))
    ax.axhline(0.5, color="0.8", ls=":")
    ax.plot(xs, t0 + t1 * xs, color=VERDE, lw=2.5, label=f"mínimos cuadrados: {errores_mc} errores")
    ax.plot(xs, log.predict_proba(xs[:, None])[:, 1], color=MORADO, lw=2.5,
            label=f"regresión logística: {errores_log} errores")
    ax.axvline(umbral_mc, color=VERDE, ls="--", lw=1.5)
    ax.axvline(umbral_log, color=MORADO, ls=":", lw=2)
    ax.scatter(x[y == 0], y[y == 0], s=110, marker="o", color=C0, ec="white", lw=1.2, zorder=3)
    ax.scatter(x[y == 1], y[y == 1], s=125, marker="^", color=C1, ec="white", lw=1.2, zorder=3)
    ax.set(xlim=(0, x_max), ylim=(-0.3, 1.5), xlabel="Tamaño del tumor", ylabel="$y$",
           title=f"Umbral de mínimos cuadrados: {umbral_mc:.2f}     Umbral de la logística: {umbral_log:.2f}")
    ax.legend(loc="upper left")
    plt.show()

In [3]:
interact(ver_tumores,
         n_lejos=widgets.IntSlider(min=0, max=3, value=0, description="Tumores grandes"),
         posicion=widgets.FloatSlider(min=10, max=40, step=0.5, value=24, description="Posición"));

interactive(children=(IntSlider(value=0, description='Tumores grandes', max=3), FloatSlider(value=24.0, descri…

## 2. Geometría de la frontera y la sigmoide (Figuras 2 y 3)

Los datos separables de la Figura 2. Escribimos los parámetros como $\theta = c\,(b, \cos\phi, \sin\phi)$: el ángulo $\phi$ fija la dirección del vector normal $(\theta_1, \theta_2)$, $b$ desplaza la frontera, y $c > 0$ multiplica todos los parámetros. Con $c = 1$, el score de un punto es exactamente su distancia con signo a la frontera.

A la izquierda, la frontera $\theta^T\mathbf{x} = 0$, el vector normal y el fondo coloreado con $h_\theta(\mathbf{x}) = g(\theta^T\mathbf{x})$. Los puntos mal clasificados aparecen con un anillo negro. A la derecha, cada punto colocado sobre la sigmoide según su score.

Qué observar:

- El vector normal es perpendicular a la frontera y apunta hacia la región donde se predice $y = 1$.
- Cambiar $b$ solo desplaza la frontera, sin girarla.
- Al subir $c$ la frontera no se mueve, pero los scores se multiplican: los puntos se alejan del centro de la sigmoide y la transición de color se vuelve más abrupta. El modelo está más seguro, aunque clasifica exactamente igual.

In [4]:
X_front = np.vstack([fig04.X0, fig04.X1])
y_front = np.concatenate([np.zeros(len(fig04.X0)), np.ones(len(fig04.X1))])
centro_front = X_front.mean(axis=0)


def ver_frontera(angulo=45, b=-3.25, c=1.0):
    phi = np.deg2rad(angulo)
    normal = np.array([np.cos(phi), np.sin(phi)])
    theta = c * np.array([b, *normal])
    z = fig04.con_unos(X_front) @ theta
    mal = (z >= 0) != (y_front == 1)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5.2), gridspec_kw=dict(width_ratios=[1, 1.1]))
    fondo(ax1, lambda P: sigmoide(fig04.con_unos(P) @ theta), (0, 5), (0, 5))
    puntos(ax1, X_front, y_front)
    ax1.scatter(*X_front[mal].T, s=320, facecolor="none", ec="black", lw=2, zorder=4)
    # Vector normal, dibujado desde el punto de la frontera más cercano al centro de los datos
    pie = centro_front - (b + normal @ centro_front) * normal
    ax1.annotate("", xy=pie + normal, xytext=pie, arrowprops=dict(arrowstyle="-|>", lw=2.5, color="black"))
    ax1.text(*(pie + 1.12 * normal), r"$(\theta_1, \theta_2)$", fontsize=13)
    ax1.set(aspect="equal", xlabel="$x_1$", ylabel="$x_2$",
            title=f"θ = ({theta[0]:.2f}, {theta[1]:.2f}, {theta[2]:.2f}):  {mal.sum()} errores")

    zs = np.linspace(-8, 8, 400)
    ax2.plot(zs, sigmoide(zs), color="black", lw=2)
    ax2.axhline(0.5, color="0.8", ls=":")
    ax2.axvline(0, color="0.8", ls=":")
    zc = np.clip(z, -8, 8)
    ax2.scatter(zc[y_front == 0], sigmoide(z[y_front == 0]), s=110, marker="o", color=C0, ec="white", lw=1.2, zorder=3)
    ax2.scatter(zc[y_front == 1], sigmoide(z[y_front == 1]), s=125, marker="^", color=C1, ec="white", lw=1.2, zorder=3)
    ax2.set(xlim=(-8, 8), ylim=(-0.05, 1.05), xlabel=r"score $z = \theta^T\mathbf{x}$", ylabel="$g(z)$",
            title="Cada punto sobre la sigmoide")
    plt.tight_layout()
    plt.show()

In [5]:
interact(ver_frontera,
         angulo=widgets.IntSlider(min=0, max=90, step=1, value=45, description="Ángulo ϕ"),
         b=widgets.FloatSlider(min=-4.5, max=-2, step=0.05, value=-3.25, description="b"),
         c=widgets.FloatLogSlider(value=1, base=10, min=-0.7, max=1.3, step=0.02, description="c"));

interactive(children=(IntSlider(value=45, description='Ángulo ϕ', max=90), FloatSlider(value=-3.25, descriptio…

## 3. Ascenso por gradiente (Figura 4)

Los 80 ejemplos de la Figura 4, cuyas clases se solapan. Para que se vea el recorrido, empezamos a propósito con un $\theta$ malo (iteración 0): la frontera casi vertical, a la derecha de los datos, y el vector normal apuntando hacia la clase 0. Ese modelo clasifica mal a la mayoría de los puntos. Desde ahí aplicamos la regla de las notas,

$$
\theta := \theta + \alpha \sum_{i=1}^m \big(y^{(i)} - h_\theta(\mathbf{x}^{(i)})\big)\,\mathbf{x}^{(i)},
$$

con $\alpha = 0.01$ y las features centradas, igual que en el script de las figuras.

Qué observar al mover la iteración:

- La frontera (negra) gira y se desplaza hasta llegar a la de la Figura 4. Las rectas punteadas son $h = 0.1$ y $h = 0.9$; siempre son paralelas a la frontera.
- En las primeras iteraciones la frontera gira rápido, porque los puntos mal clasificados tienen un error $y^{(i)} - h_\theta(\mathbf{x}^{(i)})$ grande y empujan con fuerza. Cerca del final casi todos los errores son pequeños y la frontera apenas se mueve.
- $\ell(\theta)$ sube en cada paso hasta estabilizarse en el máximo: como $\ell$ es cóncava, no hay máximos locales donde quedarse atrapado.

In [6]:
PASOS = fig04.PASOS


@lru_cache(maxsize=None)
def historia(datos, lam=0.0, alpha=0.01, inicio=(0.0, 0.0, 0.0)):
    """Parámetros en cada iteración del ascenso por gradiente, en las coordenadas originales.

    inicio es el θ de la iteración 0, en las coordenadas centradas."""
    X, y, X_original = (fig04.Xs, fig04.y_sol, fig04.X_sol) if datos == "solapan" else (fig04.Xp, fig04.y_sep, fig04.X_sep)
    thetas = np.zeros((PASOS + 1, X.shape[1]))
    theta = np.array(inicio)
    thetas[0] = theta
    for k in range(1, PASOS + 1):
        theta = theta + alpha * fig04.gradiente(theta, X, y, lam)
        thetas[k] = theta
    # Las features estaban centradas: θ_0 en las coordenadas originales es θ_0 - θ_1 μ_1 - θ_2 μ_2
    thetas[:, 0] -= thetas[:, 1:] @ X_original.mean(axis=0)
    Z = fig04.con_unos(X_original) @ thetas.T
    ell = -(y[:, None] * np.logaddexp(0, -Z) + (1 - y[:, None]) * np.logaddexp(0, Z)).sum(axis=0)
    norma = np.linalg.norm(thetas[:, 1:], axis=1)
    return thetas, ell, norma


iteraciones = np.arange(PASOS + 1)
# θ inicial malo: frontera casi vertical en x_1 ≈ 3.5 y normal apuntando hacia la clase 0
THETA_INICIAL = (1.1, -1.0, 0.3)
ell_optimo = fig04.log_verosimilitud(fig04.theta_log, fig04.con_unos(fig04.X_sol), fig04.y_sol)


def ver_ascenso(k=0):
    thetas, ell, _ = historia("solapan", inicio=THETA_INICIAL)
    theta = thetas[k]
    exactitud = np.mean((fig04.con_unos(fig04.X_sol) @ theta >= 0) == (fig04.y_sol == 1))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5.2))
    fondo(ax1, lambda P: sigmoide(fig04.con_unos(P) @ theta), (0, 5), (0, 5))
    for c in (-np.log(9), np.log(9)):  # h = 0.1 y h = 0.9
        recta(ax1, theta, c, color="0.3", ls="--", lw=1.5)
    puntos(ax1, fig04.X_sol, fig04.y_sol, tam=70)
    ax1.set(aspect="equal", xlabel="$x_1$", ylabel="$x_2$",
            title=f"Iteración {k}:  exactitud = {exactitud:.0%}")

    ax2.plot(iteraciones, ell, color=C0, lw=2.5)
    ax2.set_xscale("symlog", linthresh=1)  # escala logarítmica que incluye la iteración 0
    ax2.axhline(ell_optimo, color="0.7", ls=":", label="máximo")
    ax2.plot(k, ell[k], "o", color=C0, ms=10)
    ax2.set(xlim=(0, PASOS), xlabel="Iteración", ylabel=r"$\ell(\theta)$", title=rf"$\ell(\theta)$ = {ell[k]:.2f}")
    ax2.legend(loc="lower right")
    plt.tight_layout()
    plt.show()

In [7]:
pasos_ascenso = [0, 1, 2, 3, 4, 5, 6, 8, 10, 12, 15, 20, 25, 30, 40, 50, 70, 100, 150, 200, 300, 500, 1000, 2000, 5000, 10000, 20000]
interact(ver_ascenso, k=widgets.SelectionSlider(options=pasos_ascenso, value=0, description="Iteración"));

interactive(children=(SelectionSlider(description='Iteración', options=(0, 1, 2, 3, 4, 5, 6, 8, 10, 12, 15, 20…

## 4. Datos separables y regularización (Figura 6)

Ahora el mismo ascenso por gradiente, pero sobre los datos separables de la Figura 2.

Qué observar con $\lambda = 0$:

- La frontera deja de moverse pronto: ya separa todos los puntos.
- Pero los parámetros siguen creciendo. Las rectas $h = 0.1$ y $h = 0.9$ se pegan cada vez más a la frontera y el fondo se vuelve casi solo rojo o azul: el modelo asigna probabilidades de casi exactamente 0 o 1. A la derecha, $\|\theta\|$ (rojo) no se detiene nunca.

Qué observar al subir $\lambda$:

- Con regularización $L_2$, $\|\theta\|$ (verde) se estabiliza y las rectas punteadas se quedan a una distancia fija de la frontera. Mientras más grande $\lambda$, más pequeño $\theta$ y menos seguro el modelo.
- Como referencia, la curva azul es $\|\theta\|$ con los datos que se solapan, que se estabiliza sin regularización.

In [8]:
def ver_separables(iteracion=1, lam=0.0):
    k = int(round(iteracion))
    thetas, ell, norma = historia("separa", lam)
    theta = thetas[k]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5.2))
    fondo(ax1, lambda P: sigmoide(fig04.con_unos(P) @ theta), (0, 5), (0, 5))
    for c in (-np.log(9), np.log(9)):
        recta(ax1, theta, c, color="0.3", ls="--", lw=1.5)
    puntos(ax1, fig04.X_sep, fig04.y_sep)
    ax1.set(aspect="equal", xlabel="$x_1$", ylabel="$x_2$", title=f"Iteración {k},  λ = {lam:g}")

    ax2.semilogx(iteraciones[1:], historia("solapan")[2][1:], color=C0, lw=2, label="se solapan, λ = 0")
    ax2.semilogx(iteraciones[1:], historia("separa", 0.0)[2][1:], "--", color=C1, lw=2, label="separables, λ = 0")
    if lam > 0:
        ax2.semilogx(iteraciones[1:], norma[1:], ":", color=VERDE, lw=3, label=f"separables, λ = {lam:g}")
    ax2.plot(k, norma[k], "o", color=VERDE if lam > 0 else C1, ms=10)
    ax2.set(xlabel="Iteración", ylabel=r"$\|\theta\|$", title=rf"$\|\theta\|$ = {norma[k]:.2f}")
    ax2.legend(loc="upper left")
    plt.tight_layout()
    plt.show()

In [ ]:
interact(ver_separables,
         iteracion=widgets.FloatLogSlider(value=1, base=10, min=0, max=np.log10(PASOS), step=0.05, description="Iteración"),
         lam=widgets.SelectionSlider(options=[0.0, 0.1, 0.3, 1.0, 3.0, 10.0], value=0.0, description="λ"));

interactive(children=(FloatLogSlider(value=1.0, description='Iteración', max=4.301029995663981, step=0.05), Se…

## 5. Transformaciones de features (Figura 7)

### Una sola feature

Empezamos con el caso más simple: una sola feature $x$, con los ejemplos sobre una recta en el orden $- \; - \; + \; + \; - \; -$. Con una sola feature, la frontera de un clasificador lineal es un único punto $\theta_0 + \theta_1 x = 0$, y un punto no puede dejar a los $+$ de un lado y a todos los $-$ del otro.

Agregamos la feature $x^2$: $\phi(x) = (1, x, x^2)$. Ahora cada ejemplo vive en el plano $(x, x^2)$ y la frontera es una recta en ese plano. El slider eleva los puntos poco a poco desde la recta original ($0$) hasta su altura $x^2$ ($1$), mientras la frontera se queda fija.

Qué observar:

- En $0$ todos los puntos están en el eje y la recta no puede separarlos.
- Al elevarlos, los $-$ de los extremos suben más que los $+$ del centro, porque $x^2$ crece con la distancia al cero. Al llegar a $1$, la misma recta fija deja a los $+$ debajo (región roja) y a los $-$ encima (región azul): no movimos la frontera, movimos los puntos.
- Abajo, la recta original: la frontera en el plano corresponde a dos puntos sobre la recta, y la región de la clase 1 es el intervalo entre ellos.

In [10]:
x_1d = np.array([-3.0, -2.4, -1.9, -0.9, -0.4, 0.2, 0.7, 1.2, 2.0, 2.5, 3.1])
y_1d = np.array([0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0], dtype=float)


def phi_1d(x):
    return np.column_stack([x, x ** 2])


modelo_1d = LogisticRegression(C=10).fit(phi_1d(x_1d), y_1d)  # datos separables: un poco de regularización
theta_1d = np.concatenate([modelo_1d.intercept_, modelo_1d.coef_.ravel()])


def ver_una_feature(elevacion=0.0):
    t0, t1, t2 = theta_1d
    z = elevacion * x_1d ** 2  # altura actual de cada punto
    bien = int(np.sum((t0 + t1 * x_1d + t2 * z >= 0) == (y_1d == 1)))
    xs = np.linspace(-3.6, 3.6, 400)

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), gridspec_kw=dict(height_ratios=[3.2, 1]))
    ax1.plot(xs, elevacion * xs ** 2, color="0.85", lw=1.5, ls="--", zorder=1)
    linea = -(t0 + t1 * xs) / t2
    # θ_2 < 0: la región de la clase 1 queda debajo de la recta
    ax1.fill_between(xs, -0.8, linea, color=C1, alpha=0.12, lw=0)
    ax1.fill_between(xs, linea, 10.2, color=C0, alpha=0.08, lw=0)
    ax1.plot(xs, linea, color="black", lw=2.5, label="frontera (fija)")
    ax1.axhline(0, color="0.6", lw=0.8)
    puntos(ax1, np.column_stack([x_1d, z]), y_1d, tam=170)
    ax1.set(xlim=(-3.6, 3.6), ylim=(-0.8, 10.2), xlabel="$x$", ylabel="$x^2$" if elevacion == 1 else rf"{elevacion:.2f}$\,x^2$",
            title=f"Elevación {elevacion:.2f}:  la recta clasifica bien {bien} de {len(x_1d)} puntos")
    ax1.legend(loc="upper center")

    # Recta original: la región de la clase 1 es donde θ_0 + θ_1 x + θ_2 (elevación) x^2 >= 0
    score = t0 + t1 * xs + t2 * elevacion * xs ** 2
    ax2.fill_between(xs, -1, 1, where=score >= 0, color=C1, alpha=0.18, lw=0)
    ax2.fill_between(xs, -1, 1, where=score < 0, color=C0, alpha=0.12, lw=0)
    ax2.axhline(0, color="0.6", lw=0.8)
    puntos(ax2, np.column_stack([x_1d, np.zeros_like(x_1d)]), y_1d, tam=170)
    ax2.set(xlim=(-3.6, 3.6), ylim=(-1, 1), yticks=[], xlabel="$x$", title="Recta original")
    ax2.spines["left"].set_visible(False)
    plt.tight_layout()
    plt.show()

In [11]:
interact(ver_una_feature, elevacion=widgets.FloatSlider(min=0, max=1, step=0.02, value=0, description="Elevación"));

interactive(children=(FloatSlider(value=0.0, description='Elevación', max=1.0, step=0.02), Output()), _dom_cla…

### Parábola (datos de las notas)

Comparamos dos modelos sobre los mismos datos: la regresión logística con las features originales $(x_1, x_2)$, y la regresión logística con un feature map $\phi(\mathbf{x})$. A la izquierda, los datos y la frontera en el espacio original. A la derecha, los datos en el espacio transformado, donde la frontera del modelo con $\phi$ es una recta.

Qué observar:

- Con $(x_1, x_2)$ la frontera es una recta en el espacio original, y no puede separar las clases.
- Con $\phi$ la frontera es una curva en el espacio original, pero sigue siendo una recta en el espacio transformado: el modelo es lineal en $\theta$, y lo no lineal está en $\phi$.

En este ejemplo la clase 1 está por encima de la parábola $x_2 = x_1^2 - 0.5$ (con algo de ruido), y usamos $\phi(\mathbf{x}) = (1, x_1^2, x_2)$.

In [12]:
def ver_feature_map(X, y, phi, modelo_lin, modelo_phi, opcion, xlim, ylim, ejes_phi, tam):
    usar_phi = opcion.startswith("ϕ")
    if usar_phi:
        prob = lambda P: modelo_phi.predict_proba(phi(P))[:, 1]
        exactitud = modelo_phi.score(phi(X), y)
    else:
        prob = lambda P: modelo_lin.predict_proba(P)[:, 1]
        exactitud = modelo_lin.score(X, y)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5.2))
    fondo(ax1, prob, xlim, ylim)
    puntos(ax1, X, y, tam)
    ax1.set(xlabel="$x_1$", ylabel="$x_2$", title=f"Espacio original:  exactitud = {exactitud:.0%}")

    F = phi(X)
    puntos(ax2, F, y, tam)
    if usar_phi:
        theta = np.concatenate([modelo_phi.intercept_, modelo_phi.coef_.ravel()])
        f1 = np.linspace(F[:, 0].min() - 0.3, F[:, 0].max() + 0.3, 2)
        ax2.plot(f1, -(theta[0] + theta[1] * f1) / theta[2], color="black", lw=2)
        ax2.set_title("Espacio transformado: la frontera es una recta")
    else:
        ax2.set_title("Espacio transformado")
    ax2.set(xlim=(F[:, 0].min() - 0.3, F[:, 0].max() + 0.3), ylim=(F[:, 1].min() - 0.3, F[:, 1].max() + 0.3),
            xlabel=ejes_phi[0], ylabel=ejes_phi[1])
    ax1.legend(loc="lower right", framealpha=0.9)
    plt.tight_layout()
    plt.show()


def phi_parabola(X):
    return np.column_stack([X[:, 0] ** 2, X[:, 1]])


def ver_parabola(opcion):
    ver_feature_map(fig04.Xc, fig04.yc, phi_parabola, fig04.lin, fig04.cuad, opcion,
                    xlim=(-2.1, 2.1), ylim=(-1.1, 3.1), ejes_phi=("$x_1^2$", "$x_2$"), tam=60)

In [ ]:
interact(ver_parabola, opcion=widgets.ToggleButtons(options=["(x₁, x₂)", "ϕ(x) = (x₁², x₂)"], description="Features"));

interactive(children=(ToggleButtons(description='Features', options=('(x₁, x₂)', 'ϕ(x) = (x₁², x₂)'), value='(…

### Datos circulares

Un ejemplo que no está en las notas, con pocos puntos: la clase 1 está dentro de un círculo y la clase 0 alrededor. Ninguna recta puede dejar a la clase 1 de un lado y a la clase 0 del otro. De hecho, el mejor modelo lineal termina prediciendo todo como clase 0: no aparece ninguna frontera en la figura, y la exactitud es simplemente la fracción de puntos azules. Con $\phi(\mathbf{x}) = (1, x_1^2, x_2^2)$, la frontera $\theta_0 + \theta_1 x_1^2 + \theta_2 x_2^2 = 0$ es una elipse en el espacio original y una recta en el espacio $(x_1^2, x_2^2)$.

Como estos datos son separables, ambos modelos se entrenan con un poco de regularización ($\lambda = 0.1$).

In [14]:
rng_circ = np.random.default_rng(4)


def en_polar(radio, angulo):
    return np.column_stack([radio * np.cos(angulo), radio * np.sin(angulo)])


# Clase 1: 14 puntos dentro de un disco de radio 1.2.
# Clase 0: 18 puntos en un anillo entre radios 2.1 y 2.7, con ángulos repartidos
# parejo alrededor del círculo. El hueco entre ambos deja ver cada punto con claridad.
n_dentro, n_fuera = 14, 18
# Patrón de girasol: reparte los puntos parejo en el disco, sin que se tapen
k = np.arange(n_dentro)
dentro = en_polar(1.2 * np.sqrt((k + 0.5) / n_dentro), 2.39996 * k + rng_circ.uniform(-0.15, 0.15, n_dentro))
angulos = np.linspace(0, 2 * np.pi, n_fuera, endpoint=False) + rng_circ.uniform(-0.12, 0.12, n_fuera)
fuera = en_polar(rng_circ.uniform(2.1, 2.7, n_fuera), angulos)
X_circ = np.vstack([fuera, dentro])
y_circ = np.concatenate([np.zeros(n_fuera), np.ones(n_dentro)])


def phi_circulo(X):
    return X ** 2


lin_circ = LogisticRegression(C=10).fit(X_circ, y_circ)
phi_circ = LogisticRegression(C=10).fit(phi_circulo(X_circ), y_circ)


def ver_circulo(opcion):
    ver_feature_map(X_circ, y_circ, phi_circulo, lin_circ, phi_circ, opcion,
                    xlim=(-3.2, 3.2), ylim=(-3.2, 3.2), ejes_phi=("$x_1^2$", "$x_2^2$"), tam=200)

In [ ]:
interact(ver_circulo, opcion=widgets.ToggleButtons(options=["(x₁, x₂)", "ϕ(x) = (x₁², x₂²)"], description="Features"));

interactive(children=(ToggleButtons(description='Features', options=('(x₁, x₂)', 'ϕ(x) = (x₁², x₂²)'), value='…